In [0]:
spark.sql("SELECT current_catalog(), current_schema()").show()

In [0]:
spark.sql("SELECT ticker, COUNT(*) FROM market_intel.bronze.news_raw GROUP BY ticker").show()

In [0]:
%pip install -r /Workspace/Users/nandawritam@gmail.com/regime-market-agent/requirements-databricks.txt
dbutils.library.restartPython()

In [0]:
import sys, yaml
REPO = "/Workspace/Users/nandawritam@gmail.com/regime-market-agent"
sys.path.insert(0, REPO)
cfg = yaml.safe_load(open(f"{REPO}/config/config.yaml"))
print(cfg["tickers"]["seed"], cfg["massive"]["backfill_start_date"])

In [0]:
import requests
key = dbutils.secrets.get("capstone", "massive_api_key")
r = requests.get("https://api.massive.com/v2/reference/news",
                 params={"apiKey": key, "ticker": "NVDA",
                         "published_utc.gte": "2026-08-09T00:00:00Z", "limit": 50},
                 timeout=30)
dates = [a["published_utc"] for a in r.json()["results"]]
print(r.status_code, "| oldest:", min(dates), "| newest:", max(dates), "| n:", len(dates))

In [0]:
from src.ingestion import ingest_prices
ingest_prices.main(spark, cfg,
    secret_getter=lambda: dbutils.secrets.get("capstone", "massive_api_key"))

In [0]:
spark.sql("SELECT ticker, COUNT(*) n, MIN(source_timestamp) lo, MAX(source_timestamp) hi FROM market_intel.bronze.prices_raw GROUP BY ticker").show()
spark.sql("SELECT task, status, rows_written, error FROM market_intel.bronze.ingestion_runs ORDER BY started_at DESC LIMIT 3").show(truncate=False)

In [0]:
from src.ingestion import ingest_news
ingest_news.main(spark, cfg,
    secret_getter=lambda: dbutils.secrets.get("capstone", "massive_api_key"))